# Spark SQL: query DataFrames with real SQL at scale

_Apache Spark_

**Register a DataFrame as a view, then SELECT from it — SQL and the DataFrame API compile to the same plan, so they run identically fast.**

You already know how to read and group a table of data. The question this lesson answers is:
       what if the table is a terabyte, sitting across a cluster of machines, and the people who
       need to query it are SQL-fluent analysts, not Python programmers? Spark SQL is the answer.
       It lets you run plain structured query language (SQL) &mdash; the same
       SELECT ... WHERE ... GROUP BY ... ORDER BY you'd type against a database &mdash;
       directly on a Spark DataFrame, with Spark splitting the work across the whole cluster.

       The move is two steps. First you register a DataFrame as a named temporary view with
       df.createOrReplaceTempView("sales"). That just gives the DataFrame a name SQL can
       refer to &mdash; it copies nothing. Then you call spark.sql("SELECT ... FROM sales ...")
       and get back another DataFrame.

---

This notebook builds Spark SQL up one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — PySpark

We'll take a small `sales` table and answer one question two ways: with a real SQL string and with the DataFrame API. We build it in four steps: (1) start Spark and make the data, (2) register a view and query it with SQL, (3) run the identical query through the DataFrame API and confirm the rows match, (4) prove both compile to the same physical plan.

### Step 1 — Start Spark and build a tiny table

A `SparkSession` is the entry point to everything. Here we run in **local mode** (`local[*]`), which spins up one JVM that uses all of Colab's cores — in production this same code talks to a whole cluster instead. Then we hand-build a six-row `sales` DataFrame; in real life those rows would be terabytes read from a catalog.

In [ ]:
# Colab: !pip install pyspark   (the notebook's setup cell installs it)
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as Fsum   # alias: 'sum' is also a Python builtin

spark = (SparkSession.builder
         .master("local[*]")          # local mode: use all Colab cores, one JVM
         .appName("spark-sql-demo")
         .getOrCreate())

# A small DataFrame (in real life: terabytes read from the catalog).
rows = [
    ("books", "west", 30), ("books", "east", 20),
    ("toys",  "west", 50), ("toys",  "east",  5),
    ("games", "west", 40), ("games", "east", 15),
]
df = spark.createDataFrame(rows, schema=["category", "region", "amount"])
df.show()

### Step 2 — Register a view and query it with SQL

`createOrReplaceTempView("sales")` gives the DataFrame a name that SQL can refer to — it copies **no data**, it just registers the name in the session catalog. Once the name exists, `spark.sql(...)` runs plain SQL against it and hands back another DataFrame. The `WHERE` drops the low-value `('toys','east',5)` row before the `GROUP BY` aggregates.

In [ ]:
# Register the DataFrame as a session-scoped temporary view.
df.createOrReplaceTempView("sales")     # gives SQL a name to refer to; copies no data

# Query it with real SQL  (WHERE / GROUP BY / ORDER BY).
sql_df = spark.sql('''
    SELECT category, SUM(amount) AS total
    FROM sales
    WHERE amount >= 10                 -- drops the ('toys','east',5) row
    GROUP BY category
    ORDER BY total DESC
''')
print("SQL result:")
sql_df.show()
# +--------+-----+
# |category|total|
# +--------+-----+
# |   games|   55|   (ORDER BY total DESC)
# |   books|   50|
# |    toys|   50|
# +--------+-----+

### Step 3 — Run the identical query through the DataFrame API

The same logic — filter, group, sum, order — can be written as a chain of DataFrame method calls instead of a SQL string. Both forms describe the *same* computation. To prove it, we `collect()` each result into a set of row tuples and assert they're equal.

In [ ]:
# The identical query in the DataFrame API.
api_df = (df.filter(df.amount >= 10)
            .groupBy("category")
            .agg(Fsum("amount").alias("total"))
            .orderBy("total", ascending=False))
print("DataFrame-API result:")
api_df.show()

# Same rows? Collect both as sets of tuples and compare.
sql_rows = {tuple(r) for r in sql_df.collect()}
api_rows = {tuple(r) for r in api_df.collect()}
assert sql_rows == api_rows
print("SQL and DataFrame API returned identical rows.")

### Step 4 — Prove they compile to the same physical plan

This is the punchline: SQL strings and DataFrame calls are both handed to Spark's Catalyst optimizer, which turns them into the same **physical plan**. Calling `.explain()` on each prints that plan — they match line for line (same filter pushdown, `HashAggregate`, and `Sort`), so they run identically fast. The commented block at the end shows how you'd reach real catalog tables on a cluster, then we stop the session.

In [ ]:
# Prove they compile to the SAME physical plan.
print("---- SQL plan ----")
sql_df.explain()      # prints the physical plan
print("---- API plan ----")
api_df.explain()      # ...identical: same filter pushdown, HashAggregate, Sort

# Reading an external catalog table (Hive metastore):
# On a real cluster the catalog already knows these tables:
# spark.sql("SHOW TABLES").show()                 # list catalog tables
# orders = spark.sql("SELECT * FROM warehouse.orders")   # query a shared table
# orders2 = spark.table("warehouse.orders")              # same thing via the API
# To outlive the session, persist instead of a temp view:
# df.write.mode("overwrite").saveAsTable("warehouse.sales")

spark.stop()

## Visualize the data & results

_What does a Spark SQL 'SELECT region, SUM(amount) ... WHERE MedInc>3 GROUP BY region ORDER BY total DESC' query return — total sales by region — on a real dataset?_

### Step 1 — Load a real dataset and shape it like `sales`

A toy six-row table always behaves. Here we load the 20,640-row California housing dataset and reshape it into the same `region` / `amount` form: we bucket latitude into named regions and turn the median house value into a dollar `amount`. This gives us a realistic `sales`-style table to query.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

# Real bundled dataset: 20,640 rows of California housing.
d = fetch_california_housing(as_frame=True)
df = d.frame.copy()

# Shape it into a 'sales'-like table: a categorical 'region' and a dollar 'amount'.
lat_bins = [32, 34, 36, 38, 40, 42]
region_labels = ['far_south', 'south', 'central', 'north', 'far_north']
df['region'] = pd.cut(df['Latitude'], bins=lat_bins, labels=region_labels).astype(object)
df['amount'] = (df['MedHouseVal'] * 100000).round(0)   # median house value in dollars

### Step 2 — Run the SQL query as a pandas computation

This pandas chain mirrors the Spark SQL query *exactly*: filter to the higher-income rows (`WHERE MedInc > 3`), sum the dollar `amount` per region (`GROUP BY region`), and order the totals high to low (`ORDER BY total DESC`). It's the same query our Spark view would run, just on the local frame.

In [ ]:
# This pandas computation mirrors the Spark SQL query EXACTLY:
#   SELECT region, SUM(amount) AS total
#   FROM sales WHERE MedInc > 3
#   GROUP BY region ORDER BY total DESC
high_income = df[df['MedInc'] > 3]                       # WHERE MedInc > 3
totals = high_income.groupby('region')['amount'].sum()   # GROUP BY region, SUM(amount)
result = totals.sort_values(ascending=False)             # ORDER BY total DESC

### Step 3 — Read off the totals by region

Finally we print each region's total sales in millions of dollars, highest first — the answer the query was after. `far_south` and `central` lead; `far_north` is a rounding error by comparison.

In [ ]:
for region, total in result.items():
    print(f'{region:10s} {total/1e6:8.2f}  ($M)')
# far_south   1064.09  ($M)
# central     1055.20  ($M)
# south        898.78  ($M)
# north        250.35  ($M)
# far_north     10.32  ($M)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A colleague insists you rewrite a slow report from spark.sql("SELECT region, SUM(amount) FROM sales GROUP BY region") into the DataFrame API "because the API is faster in Spark." Is that true? How would you settle it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how Spark executes any query. — _Both SQL strings and DataFrame method calls are turned into the same logical plan, then optimized by Catalyst into the same physical plan._
- Write the API equivalent and call .explain() on both versions. — _.explain() prints the physical plan; if the plans match, the work — and therefore the runtime — is the same._
- Look for the real bottleneck instead (a wide shuffle, data skew, or a giant collect()). — _Performance comes from the plan and data movement, not from which front-end syntax produced the plan._

**Answer:** No, it's not true. SQL and the DataFrame API compile through Catalyst to the same physical plan, so they run identically fast. Prove it by calling .explain() on both forms and comparing the printed plans &mdash; they'll match. The report's slowness lives in the plan (a shuffle, skew, or a result being collected to the driver), not in the choice of SQL vs API. Rewriting the syntax changes nothing; investigate the plan and data movement instead. Keep whichever form reads better &mdash; here the SQL string is perfectly clear.

</details>

**Problem 2.** You expose a search box that filters a Spark table by category, and you implement it as spark.sql(f"SELECT * FROM products WHERE category = '{user_text}'"). Why is this risky, and what's the safer pattern?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice that user_text is pasted directly into the SQL string. — _A value like x' OR '1'='1 would rewrite the query's logic — classic SQL injection._
- Switch to the DataFrame API: spark.table("products").filter(col("category") == user_text). — _The value is passed as data, never parsed as SQL, so it cannot change the query structure._
- If you must use SQL, validate or whitelist the input first. — _Restricting input to known-safe values removes the injection surface before interpolation._

**Answer:** Interpolating user_text into the SQL string is SQL injection: a crafted value (e.g. x' OR '1'='1) can rewrite the query and leak or alter data. The safer pattern is the DataFrame API &mdash; spark.table("products").filter(col("category") == user_text) &mdash; because the value is treated as data and never parsed as SQL, so it cannot change the query's structure. If a SQL string is unavoidable, strictly validate or whitelist the input before building the string. (This is exactly where the API's programmatic composition beats raw SQL strings.)

</details>

**Problem 3.** A scheduled job runs spark.sql("SELECT * FROM sales WHERE region='west'") and fails with "Table or view 'sales' not found", even though the same query worked in your notebook yesterday. What likely happened, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how sales was created — almost certainly df.createOrReplaceTempView("sales"). — _A temporary view is session-scoped: it exists only inside the Spark session that created it._
- Note that the scheduled job runs in a fresh session. — _When the previous session ended, the temp view disappeared, so the new session has no sales to query._
- Either re-register the view at the start of the job, or persist it as a catalog table. — _Re-registering recreates the name each run; saveAsTable stores a real table that survives across sessions._

**Answer:** The sales view was registered with createOrReplaceTempView, which is session-scoped &mdash; it vanished when the notebook's session ended, so the scheduled job's fresh session has no such view. Fix it by re-registering the view inside the job (re-read the DataFrame and call createOrReplaceTempView("sales") before the query), or, if the table should outlive sessions, persist it as a real catalog table with df.write.saveAsTable("db.sales") and query that instead. Temp views are convenient but ephemeral by design.

</details>